# **Práctica y Tarea Individual: Estimación de Postura con YOLO y Reconocimiento de Actividades**

**Materia:** Visión por Computadora  
**Tema:** Estimación de Pose Humana (COCO 17 Keypoints) y Reconocimiento de Actividades / Gestos en Tiempo Real  

---
### **Objetivos de Aprendizaje**
1. Comprender qué es la estimación de postura (*pose estimation*) y cómo YOLO detecta puntos clave (*keypoints*).
2. Analizar la estructura de los 17 puntos clave del estándar **COCO Keypoints**.
3. Ejecutar inferencia en imágenes estáticas y flujos de video/cámara web.
4. **Actividad Individual (5 pts):** Implementar y presentar un sistema de reconocimiento de actividades corporales (levantar la mano, postura en cruz, sentadilla/reposo) mediante relaciones geométricas y cálculo de ángulos articulares.

### **1. Introducción a YOLO Pose Estimation**

**YOLO Pose** extiende la detección de objetos tradicional. En lugar de limitarse a predecir únicamente cajas delimitadoras (*bounding boxes*), localiza las articulaciones clave del cuerpo humano (*keypoints*) estructurando un esqueleto:

- **Salida del Modelo:** Bounding box + 17 keypoints con coordenadas `(x, y)` y confianza individual `conf`.
- **Dataset de Entrenamiento:** COCO Keypoints dataset (>200,000 imágenes anotadas con 17 articulaciones).

| Índice | Punto Clave (Keypoint) | Índice | Punto Clave (Keypoint) |
|---|---|---|---|
| **0** | Nariz (Nose) | **9** | Muñeca izquierda (L-Wrist) |
| **1** | Ojo izquierdo (L-Eye) | **10** | Muñeca derecha (R-Wrist) |
| **2** | Ojo derecho (R-Eye) | **11** | Cadera izquierda (L-Hip) |
| **3** | Oreja izquierda (L-Ear) | **12** | Cadera derecha (R-Hip) |
| **4** | Oreja derecha (R-Ear) | **13** | Rodilla izquierda (L-Knee) |
| **5** | Hombro izquierdo (L-Shoulder) | **14** | Rodilla derecha (R-Knee) |
| **6** | Hombro derecho (R-Shoulder) | **15** | Tobillo izquierdo (L-Ankle) |
| **7** | Codo izquierdo (L-Elbow) | **16** | Tobillo derecho (R-Ankle) |
| **8** | Codo derecho (R-Elbow) | | |

### **Paso 1: Instalación de Dependencias e Importaciones**

In [ ]:
# Instalación de bibliotecas necesarias (ejecutar si no se tienen instaladas)
# %pip install ultralytics opencv-python matplotlib numpy

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("Bibliotecas importadas correctamente!")

### **Paso 2: Carga del Modelo Preentrenado YOLO-Pose**
Cargamos `yolov8n-pose.pt` (versión nano preentrenada en COCO, ligera y optimizada para inferencia en tiempo real en CPU o GPU).

In [ ]:
# Cargar modelo de pose preentrenado de Ultralytics
model = YOLO('yolov8n-pose.pt')
print("Modelo YOLOv8-Pose cargado exitosamente!")

### **Paso 3: Inferencia en Imagen Estática**
Cargamos la imagen de prueba `zuck_running.jpg` y ejecutamos la predicción de puntos clave.

In [ ]:
# Cargar imagen de prueba local
img_path = 'zuck_running.jpg'
image = cv2.imread(img_path)

if image is not None:
    # Ejecutar inferencia
    results = model(image)
    result = results[0]

    # Visualizar esqueleto y cajas con el método plot()
    annotated_img = result.plot()
    
    # Convertir BGR a RGB para Matplotlib
    annotated_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 7))
    plt.imshow(annotated_rgb)
    plt.axis('off')
    plt.title('Detección de Pose con YOLOv8-Pose')
    plt.show()
else:
    print(f"No se encontró la imagen en {img_path}. Asegúrate de que el archivo exista.")

### **Paso 4: Inspección y Acceso a los Puntos Clave (Keypoints)**
Veamos las dimensiones de las predicciones y cómo extraer coordenadas específicas y niveles de confianza.

In [ ]:
if 'result' in locals() and result.keypoints is not None:
    # Forma del tensor de coordenadas: (num_personas, 17 keypoints, 2 coordenadas [x, y])
    print("Forma de keypoints.xy:", result.keypoints.xy.shape)
    
    # Coordenadas de la nariz (índice 0) de la primera persona detectada
    nariz = result.keypoints.xy[0][0].cpu().numpy()
    conf_nariz = result.keypoints.conf[0][0].cpu().numpy()
    print(f"Coordenada Nariz (x, y): {nariz} con confianza: {conf_nariz:.2f}")
    
    # Coordenadas del hombro izquierdo (índice 5) y muñeca izquierda (índice 9)
    hombro_izq = result.keypoints.xy[0][5].cpu().numpy()
    muneca_izq = result.keypoints.xy[0][9].cpu().numpy()
    print(f"Hombro Izquierdo (idx 5): {hombro_izq}")
    print(f"Muñeca Izquierda (idx 9): {muneca_izq}")

---
## **ACTIVIDAD INDIVIDUAL (Rúbrica: 5 Puntos)**

### **1. Explicación de la Actividad Seleccionada y Relevancia (1 pt)**
Hemos seleccionado un sistema de **Reconocimiento de Actividades Posturales en Tiempo Real** capaz de clasificar:
1. **Mano(s) Levantada(s)**: Identifica si el usuario levanta la mano izquierda, derecha o ambas.
   * *Relevancia:* Aplicación directa en aulas virtuales/híbridas (solicitud de palabra sin periféricos), interfaces sin contacto (HCI) y quioscos interactivos.
2. **Brazos en Cruz (T-Pose)**: Identifica cuando los brazos se encuentran extendidos horizontalmente alineados a los hombros.
   * *Relevancia:* Calibración de captura de movimiento, fisioterapia y control gestual en videojuegos o simuladores.
3. **Sentadilla / Flexión de Rodillas (Squat)**: Evalúa la flexión articular de las rodillas (cadera-rodilla-tobillo).
   * *Relevancia:* Entrenadores virtuales de fitness, conteo automático de repeticiones y monitoreo biomecánico para prevenir lesiones lumbares o de ligamentos.
4. **De Pie / Reposo**: Estado neutral de referencia.

### **2. Descripción y Código Implementado Línea por Línea (2 pts)**

In [ ]:
def calculate_angle(p1, p2, p3):
    """
    Calcula el ángulo en grados en el vértice p2 formado por los puntos p1-p2-p3.
    Utiliza álgebra vectorial: cos(theta) = (v1 . v2) / (|v1| * |v2|)
    """
    v1 = np.array([p1[0] - p2[0], p1[1] - p2[1]], dtype=np.float32)
    v2 = np.array([p3[0] - p2[0], p3[1] - p2[1]], dtype=np.float32)
    
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0:
        return 180.0
    
    cosine = np.dot(v1, v2) / (norm_v1 * norm_v2)
    cosine = np.clip(cosine, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))

def classify_pose_activity(kpts_xy, kpts_conf, conf_thresh=0.4):
    """
    Analiza las posiciones relativas y ángulos de las articulaciones
    para clasificar la actividad postural.
    """
    # Índices COCO
    NOSE, L_SHOULDER, R_SHOULDER = 0, 5, 6
    L_ELBOW, R_ELBOW = 7, 8
    L_WRIST, R_WRIST = 9, 10
    L_HIP, R_HIP = 11, 12
    L_KNEE, R_KNEE = 13, 14
    L_ANKLE, R_ANKLE = 15, 16

    # 1. Regla: Sentadilla (Ángulo de rodillas < 140 grados)
    if kpts_conf[L_HIP] > conf_thresh and kpts_conf[L_KNEE] > conf_thresh and kpts_conf[L_ANKLE] > conf_thresh:
        l_angle = calculate_angle(kpts_xy[L_HIP], kpts_xy[L_KNEE], kpts_xy[L_ANKLE])
        if l_angle < 140.0:
            return "SENTADILLA / SQUAT", (0, 140, 255), f"Rodilla Izq: {int(l_angle)} deg"
            
    if kpts_conf[R_HIP] > conf_thresh and kpts_conf[R_KNEE] > conf_thresh and kpts_conf[R_ANKLE] > conf_thresh:
        r_angle = calculate_angle(kpts_xy[R_HIP], kpts_xy[R_KNEE], kpts_xy[R_ANKLE])
        if r_angle < 140.0:
            return "SENTADILLA / SQUAT", (0, 140, 255), f"Rodilla Der: {int(r_angle)} deg"

    # Punto de referencia superior (Nariz o Hombros)
    ref_y = kpts_xy[NOSE][1] if kpts_conf[NOSE] > conf_thresh else kpts_xy[L_SHOULDER][1]

    # 2. Regla: Manos Levantadas (Muñeca sobre la nariz / hombro)
    l_up = kpts_conf[L_WRIST] > conf_thresh and (kpts_xy[L_WRIST][1] < ref_y)
    r_up = kpts_conf[R_WRIST] > conf_thresh and (kpts_xy[R_WRIST][1] < ref_y)

    if l_up and r_up:
        return "AMBAS MANOS LEVANTADAS", (0, 255, 0), "Manos arriba"
    elif r_up:
        return "MANO DERECHA LEVANTADA", (0, 220, 100), "Muñeca der arriba"
    elif l_up:
        return "MANO IZQUIERDA LEVANTADA", (0, 220, 100), "Muñeca izq arriba"

    # 3. Regla: Brazos en Cruz / T-Pose (Alineación con hombros)
    if (kpts_conf[L_SHOULDER] > conf_thresh and kpts_conf[R_SHOULDER] > conf_thresh and
        kpts_conf[L_WRIST] > conf_thresh and kpts_conf[R_WRIST] > conf_thresh):
        shoulder_y = (kpts_xy[L_SHOULDER][1] + kpts_xy[R_SHOULDER][1]) / 2.0
        if abs(kpts_xy[L_WRIST][1] - shoulder_y) < 60 and abs(kpts_xy[R_WRIST][1] - shoulder_y) < 60:
            return "BRAZOS EN CRUZ / T-POSE", (255, 200, 0), "Brazos horizontales"

    return "DE PIE / REPOSO", (200, 200, 200), "Postura normal"

### **3. Demostración en Tiempo Real con Cámara Web (2 pts)**

> **Nota para la presentación en clase:**  
> Puedes ejecutar la celda siguiente para iniciar la ventana interactiva de OpenCV con tu cámara web en vivo, o correr directamente desde la terminal el script:
> ```bash
> python 1_pose_estimation_webcam.py --source 0
> ```
> Para salir de la demostración presiona la tecla **'q'**.

In [ ]:
# Para ejecutar la demostración con webcam en vivo, descomenta y ejecuta este bloque:
"""
cap = cv2.VideoCapture(0)
print("Cámara iniciada. Presiona 'q' en la ventana para salir...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)
    annotated_frame = results[0].plot()

    if results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:
        kpts_xy = results[0].keypoints.xy[0].cpu().numpy()
        kpts_conf = results[0].keypoints.conf[0].cpu().numpy()
        actividad, color, detalle = classify_pose_activity(kpts_xy, kpts_conf)
        
        # Dibujar banner HUD superior
        cv2.rectangle(annotated_frame, (0, 0), (annotated_frame.shape[1], 60), (30, 30, 30), -1)
        cv2.putText(annotated_frame, f"ACTIVIDAD: {actividad}", (20, 40),
                    cv2.FONT_HERSHEY_DUPLEX, 0.8, color, 2)

    cv2.imshow("YOLOv8 Pose - Demostración en Vivo", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
"""
print("Código de demostración en tiempo real listo.")